# 02 · Tabular models — regression & classification
Task 1: predict kcal/100g. Task 2: classify Nutri-Score grade (note class imbalance).
Compares Random Forest / XGBoost / MLP, saves the best of each.


In [ ]:
# Make src/ importable whether on Colab or local
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[0] if (pathlib.Path.cwd().name=='notebooks') else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT/'src'))
from nutricoach import config as C


In [ ]:
from nutricoach import data, models_tabular as M, evaluate as E
df = data.load_clean()


## Task 1 — regression


In [ ]:
Xtr, Xte, ytr, yte = data.split_regression(df)
results = {}
for name, model in M.build_regressors().items():
    model.fit(Xtr, ytr)
    results[name] = E.regression_metrics(yte, model.predict(Xte))
results


In [ ]:
# pick best (lowest RMSE), refit on all data, save
best_name = min(results, key=lambda k: results[k]['RMSE'])
best = M.build_regressors()[best_name].fit(df[C.NUTRIENT_FEATURES].values, df[C.REGRESSION_TARGET].values)
M.save_model(best, M.REG_PATH); print('saved', best_name)


## Task 2 — classification (Nutri-Score)
Handle imbalance with class weights and/or SMOTE; report macro-F1, not just accuracy.


In [ ]:
Xtr, Xte, ytr, yte = data.split_classification(df)
clf_results = {}
for name, model in M.build_classifiers(class_weight='balanced').items():
    model.fit(Xtr, ytr)
    clf_results[name] = E.classification_metrics(yte, model.predict(Xte))
clf_results


In [ ]:
# SHAP feature importance for the report (which nutrients drive the grade?)
# import shap ...
